# Customer segments

Issue #7. Behavioural clusters of all 5 576 customers, then per cluster: size, spend share, churn rate, leakage. Leakage and the churn label are not clustering inputs, only profile variables, otherwise "the feeder segment leaks" would be circular.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

from src import style
from src.data import load_customer_table
from src.features import build_customer_features
from src.segments import CLUSTER_FEATURES, cluster_matrix, fit_kmeans, kmeans_scan, profile, relabel_by_spend, save_clusters

style.apply()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

feats = build_customer_features(load_customer_table())
Z, scaler = cluster_matrix(feats)
print(Z.shape, CLUSTER_FEATURES)

## features

13 columns: how much and how long (log transactions, log days active, amount per transaction, counterparts per transaction), how (e-com share, foreign share) and what (seven category shares). `total_amount`, `n_counterparts` and `n_categories` are left out, they correlate 0.87–0.96 with `n_transactions` and would triple-count activity.

In [ ]:
X = feats[CLUSTER_FEATURES].copy()
X["amt_per_tx"] = np.log1p(X["amt_per_tx"])
corr = X.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), [c.replace("cat_", "").replace("_share", "") for c in corr.columns], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(corr)), [c.replace("cat_", "").replace("_share", "") for c in corr.index], fontsize=8)
ax.grid(False)
style.clean(ax, "feature correlations (pearson)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

## how many clusters

k-means for k = 3 … 9, each with `n_init=10`. silhouette on a 2 500 sample, Davies-Bouldin on all, and stability = adjusted Rand index between seed 0 and three other seeds.

In [ ]:
scan = kmeans_scan(Z)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, col, title in zip(axes, ["silhouette", "davies_bouldin", "ari_seeds"], ["silhouette (higher is better)", "Davies-Bouldin (lower is better)", "stability across seeds (ARI)"]):
    ax.plot(scan.index, scan[col], marker="o", color=style.C[0], lw=2)
    style.clean(ax, title, "", "k")
    ax.grid(axis="x", visible=True)
plt.tight_layout()
scan.round(3)

k = 6: ARI 0.99 across seeds, Davies-Bouldin drops to 1.5, silhouette 0.20, smallest cluster 238. k = 5 is unstable (ARI 0.7), k = 7+ splits the big cluster without a new story. course guidance says 6 to 8.

In [ ]:
K = 6
km = fit_kmeans(Z, K)
labels = relabel_by_spend(km.labels_, feats)
prof = profile(feats, labels)
prof.round(3)

## the six segments

ids are renumbered by median spend, so 0 is always the biggest spender. names from the profile above.

In [ ]:
NAMES = {
    0: "main account",
    1: "travel card",
    2: "entertainment only",
    3: "online & top-ups",
    4: "subscriptions only",
    5: "tried once",
}
checks = {
    0: prof.loc[0, "n_categories"] >= 6,
    1: prof.loc[1, "cat_holidays_share"] > 0.25,
    2: prof.loc[2, "cat_entertainment_share"] > 0.5,
    3: prof.loc[3, "leak_share"] > 0.2 and prof.loc[3, "ecom_perc"] > 0.8,
    4: prof.loc[4, "cat_communication_share"] > 0.4,
    5: prof.loc[5, "n_transactions"] < 10,
}
assert all(checks.values()), f"cluster profile changed, check the names: {checks}"

table = prof[["n", "share_customers", "share_spend", "total_amount", "n_transactions", "days_active", "ecom_perc", "foreign_tx_share",
              "leak_share", "cat_savings_share", "churn_rate"]].copy()
table.index = [f"{i} · {NAMES[i]}" for i in table.index]
table.columns = ["customers", "% customers", "% spend", "median CHF", "median tx", "median days", "e-com", "foreign", "leak share", "savings share", "churn"]
table.round(2)

- **main account** (44 % of customers, 78 % of spend): median 8.8k CHF, 258 transactions, 624 days, everything in the mix. churn 32 %
- **travel card** (8 %, 11 % of spend): 7k CHF, holidays 36 % of spend, two thirds of transactions abroad. churn 36 %
- **entertainment only** (6 %): 88 % of spend in entertainment, i.e. gambling, streaming, gaming, all online. churn 68 %
- **online & top-ups** (18 %): 400 CHF, 7 transactions, 95 % e-com, a third of spend to Revolut / crypto / cash. the feeder segment. churn 73 %
- **subscriptions only** (4 %): Apple, Spotify, telco, nothing else. churn 71 %
- **tried once** (20 %): 95 CHF, 6 transactions, 37 days, a grocery shop or two. churn 88 %

three of the six segments never made YAPEAL their card. together they are 44 % of customers and 6 % of spend.

In [ ]:
z = prof[["total_amount", "n_transactions", "days_active", "amt_per_tx", "ecom_perc", "foreign_tx_share",
          "cat_groceries_share", "cat_restaurants_share", "cat_shopping_share", "cat_holidays_share", "cat_transport_share",
          "cat_entertainment_share", "cat_communication_share", "leak_share"]].copy()
z[["total_amount", "n_transactions", "days_active", "amt_per_tx"]] = np.log1p(z[["total_amount", "n_transactions", "days_active", "amt_per_tx"]])
z = (z - z.mean()) / z.std()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(z.T, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(K), [NAMES[i] for i in range(K)], rotation=20, ha="right")
ax.set_yticks(range(z.shape[1]), [c.replace("cat_", "").replace("_share", "") for c in z.columns], fontsize=8)
ax.grid(False)
style.clean(ax, "cluster profile, z-scores across clusters (red = above average)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
order = list(range(K))
for ax, col, title in zip(axes, ["churn_rate", "leak_share", "share_spend"], ["churn rate", "mean leakage share", "share of total spend"]):
    ax.bar([NAMES[i] for i in order], prof.loc[order, col] * 100, color=[style.ACCENT if i == 3 else style.C[0] for i in order], width=0.7)
    style.annotate_bars(ax, prof.loc[order, col] * 100, [f"{v * 100:.0f}" for v in prof.loc[order, col]])
    style.clean(ax, title, "%")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout()

## map

PCA to two dimensions, coloured by cluster. the first two components hold about a third of the variance, so overlap in the picture does not mean overlap in the data.

In [ ]:
pca = PCA(2, random_state=0).fit(Z)
P = pca.transform(Z)
print("explained variance:", pca.explained_variance_ratio_.round(2))

fig, ax = plt.subplots(figsize=(8, 6))
for i in range(K):
    m = labels == i
    ax.scatter(P[m, 0], P[m, 1], s=5, alpha=0.5, color=style.C[i], label=f"{NAMES[i]} (n={m.sum()})")
style.clean(ax, "customers in the first two principal components", "PC 2", "PC 1")
ax.grid(axis="x", visible=True)
ax.legend(markerscale=4, fontsize=8, loc="best")
plt.tight_layout()

## do other methods see the same segments

- HDBSCAN (`min_cluster_size=60`, `min_samples=10`) marks about 80 % of customers as noise, the data has no dense cores at this scale. on the 20 % it does cluster, it agrees with k-means (ARI 0.9)
- Ward linkage on a 1 000 customer sample for the dendrogram, cut at 6 clusters and compared to k-means on the same customers

In [ ]:
h = HDBSCAN(min_cluster_size=60, min_samples=10).fit_predict(Z)
keep = h >= 0
print(f"HDBSCAN: {h.max() + 1} clusters, {np.mean(~keep):.0%} noise, ARI vs k-means on clustered customers {adjusted_rand_score(labels[keep], h[keep]):.2f}")

rng = np.random.default_rng(0)
idx = rng.choice(len(Z), 1000, replace=False)
link = linkage(Z[idx], method="ward")
from scipy.cluster.hierarchy import fcluster
ward = fcluster(link, K, criterion="maxclust")
print(f"Ward on sample: ARI vs k-means {adjusted_rand_score(labels[idx], ward):.2f}")

fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(link, truncate_mode="lastp", p=30, ax=ax, color_threshold=0, above_threshold_color=style.C[0], no_labels=True)
style.clean(ax, "Ward linkage, 1 000 customer sample, last 30 merges", "distance")
plt.tight_layout()

HDBSCAN agrees where it clusters at all. Ward finds the same big segment and cuts the small online ones differently (ARI 0.4). the segments are directions, not walls, which is fine for a story and for targeting.

## what YAPEAL could do with them

| segment | one action |
|---|---|
| main account | keep, these are the bank. retention triggers only when their leakage rises |
| travel card | make it the main account: salary or standing order nudge after a trip, FX at fair rates |
| entertainment only | responsible-banking check, gambling limits, not a growth segment |
| online & top-ups | the feeder segment. interest on balances or an in-app savings/crypto product, otherwise they leave for Revolut |
| subscriptions only | low value, low cost, leave alone |
| tried once | onboarding problem, not a retention problem. first-week activation, not offers |

In [ ]:
out = save_clusters(feats, labels, NAMES)
print("written data/processed/customer_clusters.csv")
out["cluster_name"].value_counts()